# **1. Análise exploratória**

Antes de qualquer modelo, este notebook observa como o público varia com cada fator candidato. O tamanho do clube domina o público bruto (o público do Flamengo é várias vezes o do Juventude), então todas as comparações usam o **público relativo**: o público dividido pela média móvel dos últimos cinco públicos regulares do clube em casa. Um valor de +10% indica um público 10% acima do que o clube vinha levando.

| tabela | pergunta |
|---|---|
| nível dos clubes | qual é o tamanho e a variação do público de cada clube? |
| público relativo por condição | quais dias, horários, climas, adversários e estádios alteram o público? |
| público por estádio | qual é o público de cada clube em cada um de seus estádios? |
| jogos com restrição | quais jogos foram disputados com restrição de público? |

As tabelas são exportadas para `data/processed/exploratory/`; as figuras feitas a partir delas, com a discussão, estão no notebook 06.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

### **1.1. Configuração**

O público relativo só é calculado quando o clube tem ao menos cinco públicos regulares anteriores em casa, para que a média móvel tenha a janela completa de cinco jogos. Os intervalos são intervalos bootstrap de 90% para a mediana.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
OUTPUT = PROCESSED / "exploratory"
OUTPUT.mkdir(parents=True, exist_ok=True)

N_BOOT = 2000
SEED = 7
rng = np.random.default_rng(SEED)

features = pd.read_csv(PROCESSED / "features.csv", parse_dates=["date", "kickoff_local"])
regular = features[features["crowd_status"].eq("regular")].copy()
relative = regular[regular["att_history_n"].ge(5)].copy()
relative["relative"] = relative["attendance"] / relative["att_ma5"] - 1
print(f"jogos regulares: {len(regular):,} | com média móvel completa: {len(relative):,}")

jogos regulares: 1,675 | com média móvel completa: 1,525


## **2. Nível dos clubes**

Público mediano de cada clube nos jogos em casa da Série A depois da pandemia, com o intervalo interquartil. Clubes com menos de dez jogos regulares em casa ficam de fora.

In [3]:
levels = regular.groupby("home")["attendance"].agg(matches="size", q25=lambda s: s.quantile(0.25), median="median", q75=lambda s: s.quantile(0.75))
levels["seasons"] = regular.groupby("home")["season"].nunique()
levels = levels[levels["matches"].ge(10)].sort_values("median")

display(levels.round(0).sort_values("median", ascending=False).head(10))

,matches,q25,median,q75,seasons
home,,,,,
Flamengo,83,53410.0,60642.0,64649.0,5
Corinthians,85,37237.0,41118.0,43996.0,5
Bahia,65,30960.0,35993.0,42022.0,4
Cruzeiro,64,23518.0,35915.0,44747.0,4
Palmeiras,83,29112.0,35835.0,39706.0,5
São Paulo,84,24562.0,34916.0,46352.0,5
Fortaleza,76,22435.0,30498.0,39426.0,4
Atlético-MG,82,24549.0,29569.0,36244.0,5
Ceará,36,24476.0,29568.0,40994.0,2


## **3. Público relativo por condição**

Cada linha compara um conjunto de jogos com o nível recente do próprio clube. Os conjuntos vêm das famílias de variáveis: o dia e o horário do jogo, o clima, o momento esportivo, o adversário e o estádio.

In [4]:
def kickoff_band(hour):
    return "antes das 17h" if hour < 17 else "das 17h às 19h59" if hour < 20 else "20h ou mais tarde"


relative["weekday_label"] = relative["weekday"].map(dict(enumerate(["segunda", "terça", "quarta", "quinta", "sexta", "sábado", "domingo"])))
relative["kickoff_band"] = relative["kickoff_hour"].map(kickoff_band)
relative["temperature_band"] = pd.cut(relative["temperature"], [-np.inf, 18, 26, np.inf], labels=["abaixo de 18 °C", "de 18 a 26 °C", "acima de 26 °C"])
relative["form_band"] = pd.cut(relative["ppg_last5"], [-np.inf, 1.0, 2.0, np.inf], labels=["menos de 1 ponto por jogo", "de 1 a 2 pontos por jogo", "mais de 2 pontos por jogo"])

CONDITIONS = [
    ("Dia da semana", "weekday_label", ["segunda", "terça", "quarta", "quinta", "sexta", "sábado", "domingo"]),
    ("Horário", "kickoff_band", ["antes das 17h", "das 17h às 19h59", "20h ou mais tarde"]),
    ("Feriado", "holiday", [1, 0]),
    ("Chuva", "rainy", [1.0, 0.0]),
    ("Temperatura", "temperature_band", ["abaixo de 18 °C", "de 18 a 26 °C", "acima de 26 °C"]),
    ("Momento (últimos 5 jogos)", "form_band", ["menos de 1 ponto por jogo", "de 1 a 2 pontos por jogo", "mais de 2 pontos por jogo"]),
    ("Adversário", "away_big_club", [1, 0]),
    ("Clássico", "derby_state", [1, 0]),
    ("Estádio", "venue_usual", [0, 1]),
]
LABELS = {
    ("holiday", 1): "feriado", ("holiday", 0): "dia comum", ("rainy", 1.0): "chuva nas 4h antes do jogo", ("rainy", 0.0): "tempo seco",
    ("away_big_club", 1): "visita de um dos 12 grandes", ("away_big_club", 0): "outro visitante", ("derby_state", 1): "rival do mesmo estado",
    ("derby_state", 0): "visitante de outro estado", ("venue_usual", 0): "fora do estádio habitual", ("venue_usual", 1): "estádio habitual",
}


def bootstrap_median(values: np.ndarray) -> tuple[float, float, float]:
    samples = rng.choice(values, size=(N_BOOT, len(values)), replace=True)
    medians = np.median(samples, axis=1)
    return float(np.median(values)), float(np.quantile(medians, 0.05)), float(np.quantile(medians, 0.95))


condition_rows = []
for family, column, levels_order in CONDITIONS:
    for level in levels_order:
        values = relative.loc[relative[column] == level, "relative"].dropna().to_numpy()
        if len(values) < 15:
            continue
        median, low, high = bootstrap_median(values)
        condition_rows.append({"family": family, "level": LABELS.get((column, level), str(level)), "matches": len(values), "median": median, "low": low, "high": high})
conditions = pd.DataFrame(condition_rows)
display(conditions.round(3))

,family,level,matches,median,low,high
0,Dia da semana,segunda,62,-0.025,-0.131,0.039
1,Dia da semana,terça,39,-0.117,-0.157,-0.008
2,Dia da semana,quarta,274,-0.156,-0.207,-0.104
3,Dia da semana,quinta,124,-0.083,-0.118,-0.010
4,Dia da semana,sexta,17,-0.035,-0.272,0.071
5,Dia da semana,sábado,428,0.002,-0.044,0.019
6,Dia da semana,domingo,581,0.060,0.029,0.081
7,Horário,antes das 17h,393,0.093,0.077,0.114
8,Horário,das 17h às 19h59,636,-0.057,-0.092,-0.024
9,Horário,20h ou mais tarde,496,-0.071,-0.109,-0.041


## **4. Público por estádio**

Para cada clube, o público mediano no estádio habitual é comparado com o público mediano em cada outro estádio onde mandou ao menos dois jogos regulares. Uma razão acima de um indica que o clube levou mais público fora do estádio habitual, como o Vasco no Maracanã.

In [5]:
usual_venue = regular.groupby(["home", "season"])["stadium"].agg(lambda s: s.mode().iat[0]).rename("usual_stadium")
moves = regular.join(usual_venue, on=["home", "season"])
usual_level = moves[moves["stadium"].eq(moves["usual_stadium"])].groupby("home").agg(usual_stadium=("stadium", lambda s: s.mode().iat[0]), usual_median=("attendance", "median"))
other = moves[moves["venue_usual"].eq(0)].groupby(["home", "stadium"]).agg(matches=("game_id", "size"), median=("attendance", "median"), seasons=("season", lambda s: ", ".join(map(str, sorted(s.unique())))))
stadium_moves = other[other["matches"].ge(2)].reset_index().join(usual_level, on="home")
stadium_moves = stadium_moves[stadium_moves["stadium"].ne(stadium_moves["usual_stadium"])]
stadium_moves["ratio"] = stadium_moves["median"] / stadium_moves["usual_median"]
stadium_moves = stadium_moves.sort_values("ratio")
display(stadium_moves.round({"median": 0, "usual_median": 0, "ratio": 2}))

,home,stadium,matches,median,seasons,usual_stadium,usual_median,ratio
9,Fluminense,Raulino de Oliveira,5,7788.0,"2022, 2023",Maracanã,30397.0,0.26
22,São Paulo,Vila Belmiro,5,10086.0,"2023, 2025",Morumbi,36764.0,0.27
10,Fortaleza,Presidente Vargas,3,12467.0,"2023, 2024, 2025",Arena Castelão,30822.0,0.40
21,São Paulo,Brinco de Ouro,2,15602.0,"2024, 2026",Morumbi,36764.0,0.42
17,Palmeiras,Brinco de Ouro,2,16303.0,2024,Nubank Parque,38146.0,0.43
16,Palmeiras,Arena Barueri,12,17175.0,"2022, 2023, 2024, 2025, 2026",Nubank Parque,38146.0,0.45
12,Grêmio,Centenário,4,11395.0,2024,Arena do Grêmio,24213.0,0.47
5,Cruzeiro,Independência,2,20448.0,2024,Mineirão,36882.0,0.55
2,Atlético-MG,Independência,4,17506.0,"2022, 2023",ARENA MRV,30003.0,0.58
14,Internacional,Heriberto Hulse,2,14522.0,2024,Beira-Rio,23604.0,0.62


## **5. Jogos com restrição**

Jogos marcados no notebook 01 como disputados com restrição. Eles ficam fora dos alvos de treino e do histórico.

In [6]:
restricted = features[features["crowd_status"].eq("restricted")][["season", "round", "date", "home", "away", "stadium", "attendance", "att_ma5"]]
display(restricted.sort_values("date"))
display(features.groupby(["season", "crowd_status"]).size().unstack(fill_value=0))

,season,round,date,home,away,stadium,attendance,att_ma5
50,2022,6,2022-05-14,Palmeiras,Red Bull Bragantino,Nubank Parque,2599.0,26616.333333
750,2023,38,2023-12-06,Goiás,América Mineiro,Hailé Pinheiro,1966.0,11350.000000


crowd_status,missing,regular,restricted
season,,,
2022,4,375,1
2023,12,367,1
2024,4,376,0
2025,0,380,0
2026,183,177,0


## **6. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/exploratory/club_levels.csv` | mediana e intervalo interquartil do público de cada clube |
| `data/processed/exploratory/relative_crowd_by_condition.csv` | mediana do público relativo e intervalo por condição |
| `data/processed/exploratory/stadium_moves.csv` | público mediano de cada clube no estádio habitual e nos outros estádios |

In [7]:
levels.to_csv(OUTPUT / "club_levels.csv")
conditions.to_csv(OUTPUT / "relative_crowd_by_condition.csv", index=False)
stadium_moves.to_csv(OUTPUT / "stadium_moves.csv", index=False)
assert relative["relative"].notna().all(), "Público relativo com média móvel ausente"
assert conditions["low"].le(conditions["median"]).all() and conditions["median"].le(conditions["high"]).all(), "O intervalo não contém a mediana"
assert stadium_moves["ratio"].gt(0).all(), "Razão de estádio inválida"